# ViMed: Qwen3-Embedding-8B cho 226k doc zh (kho mới)
Chuẩn bị: upload 5 file trong `runs/colab_zh_q8/` lên Google Drive, thư mục **`MyDrive/vimed_colab/`**.
Runtime → Change runtime type → **A100 GPU**. Chạy lần lượt từ trên xuống.
Mã hoá lưu từng phần (4 phần) vào Drive ⇒ mất kết nối thì chạy lại, phần đã xong được bỏ qua.
Kết quả: 3 file `dense_zhQ8_*_rank.jsonl` trong `MyDrive/vimed_colab/` → tải về `vimed/data/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
D = '/content/drive/MyDrive/vimed_colab'; W = '/content/w'
os.makedirs(W, exist_ok=True)
for f in ['dense.py', 'zh_text_new.sqlite', 'queries.jsonl', 'queries_zhq.jsonl', 'queries_zhq109.jsonl']:
    shutil.copy(f'{D}/{f}', W)
print(os.listdir(W))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install -U "sentence-transformers>=3.0" "transformers>=4.51"

In [ ]:
import sys, time, json, sqlite3, numpy as np
sys.path.insert(0, W)
import dense
M = 'Qwen/Qwen3-Embedding-8B'
model = dense.load_model(name=M)        # fp16, padding_side=left
model.max_seq_length = 256              # như bản BGE-M3 (title + 600 ký tự, cắt 256 token)

In [ ]:
# Mã hoá 4 phần, mỗi phần lưu ngay vào Drive (an toàn khi mất kết nối)
src = f'{W}/zh_text_new.sqlite'
for k in range(4):
    out = f'{D}/emb_zhQ8_s{k}'
    if os.path.exists(out + '.npy'):
        print('bỏ qua phần', k); continue
    p = f'{W}/part{k}.sqlite'
    if os.path.exists(p): os.remove(p)
    c = sqlite3.connect(p); c.execute('create table doc(doc_id text, title text, body text)')
    c.execute(f"attach '{src}' as s"); c.execute('insert into doc select doc_id,title,body from s.doc where rowid % 4 = ?', (k,)); c.commit(); c.close()
    t = time.time()
    dense.encode_docs(p, out, batch=256, model=model, min_title=3, min_body=0, body_chars=600)
    print(f'phần {k}: {time.time()-t:.0f}s')

In [ ]:
E = [np.load(f'{D}/emb_zhQ8_s{k}.npy') for k in range(4)]
I = [json.load(open(f'{D}/emb_zhQ8_s{k}.ids.json')) for k in range(4)]
np.save(f'{W}/emb_zhQ8.npy', np.concatenate(E)); json.dump(sum(I, []), open(f'{W}/emb_zhQ8.ids.json', 'w'))
print('emb_zhQ8', sum(map(len, I)))

In [ ]:
INS_VI = 'Instruct: Given a Vietnamese patient health question, retrieve Chinese medical Q&A posts or articles that answer it\nQuery:'
INS_ZH = 'Instruct: Given a patient health question, retrieve medical Q&A posts or articles that answer it\nQuery:'
for qf, tag, ins in [('queries.jsonl', 'vi', INS_VI), ('queries_zhq.jsonl', 'zhq', INS_ZH), ('queries_zhq109.jsonl', 'zhq109', INS_ZH)]:
    rows = dense.rank(f'{W}/emb_zhQ8', dense.read_queries(f'{W}/{qf}'), topk=1000, model=model, prompt=ins)
    dense.write_jsonl(f'{D}/dense_zhQ8_{tag}_rank.jsonl', rows)
print('XONG - tải 3 file dense_zhQ8_*_rank.jsonl trong MyDrive/vimed_colab về vimed/data/')